# Run Experiments

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/polynormer-reimplementation

In [ ]:
!pip install -r requirements.txt

In [ ]:
import gc
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch

# Get current disctory
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.polynormer import Polynormer
from train import evaluate
from utils.data_loaders import load_dataset
from utils.io import load_checkpoint

# Define static variables
PYTHON_EXECUTABLE = sys.executable
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_CSV_PATH = ROOT / "experiment_results.csv"

In [ ]:
COMMON_SETTINGS = {
    "hidden_dim": 512,
    "lr": 1e-3,
    "n_local_heads": 8,
    "n_global_heads": 8,
    "weight_decay": 0.0,
}

DATASET_CONFIGS = {
    "computer": {
        "name": "Computer",
        "metric": "accuracy",
        "params": {
            "warm_up_epochs": 200,
            "local_to_global_epochs": 1000,
            "n_local_layers": 5,
            "n_global_layers": 1,
            "dropout": 0.7,
        },
        "baseline": (93.18, 0.18),
        "baseline_relu": (93.68, 0.21),
    },
    "photo": {
        "name": "Photo",
        "metric": "accuracy",
        "params": {
            "warm_up_epochs": 200,
            "local_to_global_epochs": 1000,
            "n_local_layers": 7,
            "n_global_layers": 2,
            "dropout": 0.7,
        },
        "baseline": (96.11, 0.23),
        "baseline_relu": (96.46, 0.26),
    },
    "cs": {
        "name": "CS",
        "metric": "accuracy",
        "params": {
            "warm_up_epochs": 100,
            "local_to_global_epochs": 1500,
            "n_local_layers": 5,
            "n_global_layers": 2,
            "dropout": 0.3,
        },
        "baseline": (95.51, 0.29),
        "baseline_relu": (95.53, 0.16),
    },
    "physics": {
        "name": "Physics",
        "metric": "accuracy",
        "params": {
            "warm_up_epochs": 100,
            "local_to_global_epochs": 500,
            "n_local_layers": 5,
            "n_global_layers": 4,
            "dropout": 0.5,
        },
        "baseline": (97.22, 0.06),
        "baseline_relu": (97.27, 0.08),
    },
    "wikics": {
        "name": "WikiCS",
        "metric": "accuracy",
        "params": {
            "warm_up_epochs": 100,
            "local_to_global_epochs": 1000,
            "n_local_layers": 7,
            "n_global_layers": 2,
            "dropout": 0.5,
        },
        "baseline": (79.53, 0.83),
        "baseline_relu": (80.10, 0.67),
    },
    "roman-empire": {
        "name": "roman-empire",
        "metric": "accuracy",
        "params": {
            "warm_up_epochs": 100,
            "local_to_global_epochs": 2500,
            "n_local_layers": 10,
            "n_global_layers": 2,
            "dropout": 0.3,
        },
        "baseline": (92.13, 0.50),
        "baseline_relu": (92.55, 0.37),
    },
    "amazon-ratings": {
        "name": "amazon-ratings",
        "metric": "accuracy",
        "params": {
            "warm_up_epochs": 200,
            "local_to_global_epochs": 2500,
            "n_local_layers": 10,
            "n_global_layers": 1,
            "dropout": 0.3,
        },
        "baseline": (54.46, 0.40),
        "baseline_relu": (54.81, 0.49),
    },
    "minesweeper": {
        "name": "minesweeper",
        "metric": "roc_auc",
        "params": {
            "warm_up_epochs": 100,
            "local_to_global_epochs": 2000,
            "n_local_layers": 10,
            "n_global_layers": 3,
            "dropout": 0.3,
        },
        "baseline": (96.96, 0.52),
        "baseline_relu": (97.46, 0.36),
    },
    "tolokers": {
        "name": "tolokers",
        "metric": "roc_auc",
        "params": {
            "warm_up_epochs": 100,
            "local_to_global_epochs": 800,
            "n_local_layers": 7,
            "n_global_layers": 2,
            "dropout": 0.5,
        },
        "baseline": (84.83, 0.72),
        "baseline_relu": (85.91, 0.74),
    },
    "questions": {
        "name": "questions",
        "metric": "roc_auc",
        "params": {
            "warm_up_epochs": 200,
            "local_to_global_epochs": 1500,
            "n_local_layers": 5,
            "n_global_layers": 3,
            "dropout": 0.2,
        },
        "baseline": (77.95, 1.06),
        "baseline_relu": (78.92, 0.89),
    },
    "ogbn-arxiv": {
        "name": "ogbn-arxiv",
        "metric": "accuracy",
        "params": {
            "warm_up_epochs": 2000,
            "local_to_global_epochs": 500,
            "n_local_layers": 7,
            "n_global_layers": 2,
            "dropout": 0.5,
        },
        "baseline": (71.82, 0.23),
        "baseline_relu": (73.46, 0.16),
    },
}

DEFAULT_SEED = 42

In [ ]:
# Build model for evaluation
def build_model(configuration, in_dim, out_dim, device):
    return Polynormer(
        in_dim=in_dim,
        hidden_dim=configuration["hidden_dim"],
        out_dim=out_dim,
        n_local_layers=configuration["n_local_layers"],
        n_global_layers=configuration["n_global_layers"],
        n_local_heads=configuration["n_local_heads"],
        n_global_heads=configuration["n_global_heads"],
        dropout=configuration["dropout"],
        use_relu=configuration["use_relu"],
        use_local_attention_network=configuration["use_local_attention_network"],
    ).to(device)


# Match the checkpoint naming used inside train.py
def get_checkpoint_path(dataset_name, use_relu):
    name = f"checkpoint_{dataset_name.lower()}_{'relu' if use_relu else 'plain'}"
    return ROOT / "checkpoints" / name


# Build command to be executed in the terminal to train the models
def build_train_command(configuration):
    args = [
        PYTHON_EXECUTABLE,
        str(ROOT / "train.py"),
        "--dataset",
        configuration["dataset"],
        "--hidden_dim",
        str(configuration["hidden_dim"]),
        "--n_local_layers",
        str(configuration["n_local_layers"]),
        "--n_global_layers",
        str(configuration["n_global_layers"]),
        "--n_local_heads",
        str(configuration["n_local_heads"]),
        "--n_global_heads",
        str(configuration["n_global_heads"]),
        "--warm_up_epochs",
        str(configuration["warm_up_epochs"]),
        "--local_to_global_epochs",
        str(configuration["local_to_global_epochs"]),
        "--use_relu",
        str(configuration["use_relu"]),
        "--use_local_attention_network",
        str(configuration["use_local_attention_network"]),
        "--lr",
        str(configuration["lr"]),
        "--dropout",
        str(configuration["dropout"]),
        "--metric",
        configuration["metric"],
    ]
    return args


# Convert command to a string
def command_to_string(command):
    return shlex.join(command)


# Evaluate a checkpoint
def evaluate_checkpoint(configuration, checkpoint_path, device):
    data, in_dim, out_dim = load_dataset(
        configuration["dataset"],
        root="data",
        seed=configuration["seed"],
    )

    # Recover model from training
    model = build_model(configuration, in_dim, out_dim, device)
    load_checkpoint(str(checkpoint_path), model)

    metrics = evaluate(
        model,
        data,
        device,
        freeze_global=False,
        metric=configuration["metric"],
    )
    return {
        "best_val_metric": float(metrics["val_acc"] * 100.0),
        "best_test_metric": float(metrics["test_acc"] * 100.0),
    }

In [ ]:
def format_new_result(result):
    if result["status"] != "ok":
        return f"{result['status']}: {result.get('error', 'unknown error')}"
    return f"{result['best_test_metric']:.2f}"


def format_baseline_result(result):
    mean, std = result
    return f"{mean:.2f}±{std:.2f}"


# Combine common hyperparameters and dataset configurations
def build_configuration(dataset_name, use_relu):
    spec = DATASET_CONFIGS[dataset_name]
    params = spec["params"]

    return {
        "dataset": dataset_name,
        "seed": DEFAULT_SEED,
        "metric": spec["metric"],
        "hidden_dim": COMMON_SETTINGS["hidden_dim"],
        "n_local_layers": params["n_local_layers"],
        "n_global_layers": params["n_global_layers"],
        "n_local_heads": COMMON_SETTINGS["n_local_heads"],
        "n_global_heads": COMMON_SETTINGS["n_global_heads"],
        "warm_up_epochs": params["warm_up_epochs"],
        "local_to_global_epochs": params["local_to_global_epochs"],
        "use_relu": use_relu,
        "use_local_attention_network": False if dataset_name == "ogbn-arxiv" else True,
        "lr": COMMON_SETTINGS["lr"],
        "dropout": params["dropout"],
        "weight_decay": COMMON_SETTINGS["weight_decay"],
        "checkpoint_path": str(get_checkpoint_path(dataset_name, use_relu)),
    }


def run_training(dataset_name, use_relu, device):
    configuration = build_configuration(dataset_name, use_relu)
    checkpoint_path = Path(configuration["checkpoint_path"])
    command = build_train_command(configuration)

    try:
        if checkpoint_path.exists():
            print(f"Using existing checkpoint: {checkpoint_path}")
        else:
            _ = subprocess.run(
                command, cwd=str(ROOT), capture_output=True, text=True, check=False
            )

        metrics = evaluate_checkpoint(configuration, checkpoint_path, device)
        return {
            "status": "ok",
            "best_val_metric": metrics["best_val_metric"],
            "best_test_metric": metrics["best_test_metric"],
        }

    except RuntimeError as exc:
        message = str(exc)
        if "out of memory" in message.lower():
            return {
                "status": "oom",
                "error": message,
                "command": command_to_string(command),
            }
        return {
            "status": "runtime_error",
            "error": message,
            "command": command_to_string(command),
        }
    except Exception as exc:
        return {
            "status": "train_failed",
            "error": str(exc),
            "command": command_to_string(command),
        }
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()


# Summarize the result
def summarize_result(dataset_name, result, relu_result):
    spec = DATASET_CONFIGS[dataset_name]

    summary = {
        "dataset": dataset_name,
        "metric": "Accuracy" if spec["metric"] == "accuracy" else "ROC-AUC",
        "baseline": format_baseline_result(spec["baseline"]),
        "new_result": format_new_result(result),
        "baseline_relu": format_baseline_result(spec["baseline_relu"]),
        "new_result_relu": format_new_result(relu_result),
    }

    print(
        "=== Training Completed === \n",
        f"Metric: {summary["metric"]} | ",
        f"Result vs. Baseline: {summary["new_result"]} vs. {summary["baseline"]} | ",
        f"Result vs. Baseline (ReLU): {summary["new_result_relu"]} vs. {summary["baseline_relu"]}",
    )
    return {
        "dataset": dataset_name,
        "metric": "Accuracy" if spec["metric"] == "accuracy" else "ROC-AUC",
        "baseline": format_baseline_result(spec["baseline"]),
        "new_result": format_new_result(result),
        "baseline_relu": format_baseline_result(spec["baseline_relu"]),
        "new_result_relu": format_new_result(relu_result),
    }


def build_results_table(records):
    return pd.DataFrame(records)

In [ ]:
RUN_DATASETS = list(DATASET_CONFIGS)
SAVE_RESULTS_CSV = True

print(f"Device: {DEVICE}")
print(f"Seed: {DEFAULT_SEED}")
print(f"Datasets: {RUN_DATASETS}")

In [ ]:
records = []

for dataset_name in RUN_DATASETS:
    print(f"\n=== {dataset_name} ===")
    result = run_training(dataset_name=dataset_name, use_relu=False, device=DEVICE)
    relu_result = run_training(dataset_name=dataset_name, use_relu=True, device=DEVICE)
    records.append(summarize_result(dataset_name, result, relu_result))

results_table = build_results_table(records)
print(results_table)

In [ ]:
ordered_columns = [
    "dataset",
    "metric",
    "baseline",
    "new_result",
    "baseline_relu",
    "new_result_relu",
]
results_table = results_table[ordered_columns]
display(results_table)

if SAVE_RESULTS_CSV:
    results_table.to_csv(RESULTS_CSV_PATH, index=False)
    print(f"Saved CSV to: {RESULTS_CSV_PATH}")